# 09 · Federated platinum models → local NEPC / AVPC

Wrapper notebook. All logic lives in `survival_common/federated_endpoint_transfer.py`
(CLI equivalent: `python -m survival_common.federated_endpoint_transfer`).

The federated bundles are XGBoost and elastic-net Cox models trained on **time to platinum**
in the caia-project-compass `rhino_scripts` environments. This notebook scores those frozen models
on the full local DFCI ADT cohort and checks how well they predict the fine-grained LLM endpoints
**time to NEPC** and **time to AVPC**. Platinum is kept as a same-endpoint reference.

The pooled preprocessing in each bundle (impute means, centers and scales) is applied as shipped
and never refit locally. Every endpoint shares the same feature frame, so each patient's risk score
is the same for every endpoint; only the outcome changes.

**Metrics** for each endpoint × landmark × bundle × config:
- Harrell's C with a patient-bootstrap 95% CI.
- Paired ΔC vs the bundle's age-only `baseline` config.
- Cox HR per SD of risk score.
- Uno's C and cumulative/dynamic AUC(t). Censoring is estimated on the local cohort.

## Configuration

In [ ]:
import os, sys, json
from pathlib import Path

sys.path.insert(0, ".")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from survival_common.federated_inference import load_bundle
from survival_common import federated_endpoint_transfer as fet

DATA_ROOT = cp._PROFILE_OUTPUT_ROOT

# >>> Edit: directory holding the downloaded federated model bundles (*.json) <<<
FEDERATED_MODELS_DIR = Path(DATA_ROOT) / "federated_models"
BUNDLE_GLOB = "*.json"          # or list explicit files in BUNDLE_PATHS below

DATA_PATH   = Path(DATA_ROOT) / "longitudinal_prediction_data_adt.csv"
OUTPUT_DIR  = Path(DATA_ROOT) / "federated_transfer_adt"
LAB_MAPPING = fet.DEFAULT_LAB_MAPPING

ENDPOINTS   = ("nepc", "avpc", "platinum")
EXCLUSION   = "none"            # "none" = full ADT cohort; "pre_anchor_castrate" for noprecastrate bundles
LANDMARKS   = None              # None = every landmark present in the bundles
HORIZONS    = fet.DEFAULT_HORIZONS_DAYS   # AUC(t) horizons, days after the landmark
N_BOOTSTRAP = 200
SEED        = 0

BUNDLE_PATHS = sorted(FEDERATED_MODELS_DIR.glob(BUNDLE_GLOB))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(BUNDLE_PATHS)} bundle(s) in {FEDERATED_MODELS_DIR}")
for p in BUNDLE_PATHS:
    print("  ", p.name)

## Load bundles

In [ ]:
assert BUNDLE_PATHS, f"no bundles matching {BUNDLE_GLOB!r} in {FEDERATED_MODELS_DIR}"

bundles = {}
rows = []
for path in BUNDLE_PATHS:
    b = load_bundle(path)
    bundles[path.stem] = b
    label = str(b.get("analysis_label") or "")
    if ("noprecastrate" in label) != (EXCLUSION == "pre_anchor_castrate"):
        print(f"WARNING: {path.stem} analysis_label={label!r} but EXCLUSION={EXCLUSION!r}")
    for m in b["models"]:
        rows.append({
            "bundle": path.stem, "model_family": b.get("model_family"),
            "analysis_label": label, "trained_endpoint": b.get("endpoint", "platinum"),
            "landmark_days": m["landmark_days"], "config": m.get("config"),
            "n_covariates": len(m["covariate_cols"]),
        })
pd.DataFrame(rows)

## Local cohort

In [ ]:
local = fet.read_local_frame(DATA_PATH, ENDPOINTS)
lab_mapping = fet.load_lab_mapping(LAB_MAPPING)

pt = local.drop_duplicates(fet.LOCAL_ID_COL)
print(f"{len(local):,} rows, {len(pt):,} patients from {DATA_PATH}")
pd.DataFrame({
    ep: {"events": int(pt[ind].fillna(0).astype(int).sum()),
         "dated events": int((pt[ind].fillna(0).astype(int).eq(1) & pt[dt].notna()).sum())}
    for ep, (ind, dt) in fet.ENDPOINT_SOURCES.items() if ep in ENDPOINTS
}).T

## Run transfer evaluation

In [ ]:
res = fet.evaluate_transfer(
    local, bundles,
    endpoints=ENDPOINTS, lab_mapping=lab_mapping, exclusion=EXCLUSION,
    landmarks=LANDMARKS, horizons_days=HORIZONS,
    n_bootstrap=N_BOOTSTRAP, seed=SEED,
)

for name in ("scores", "metrics", "auc_t", "coverage", "unit_check"):
    if not res[name].empty:
        res[name].to_csv(OUTPUT_DIR / f"transfer_{name}.csv", index=False)
with open(OUTPUT_DIR / "transfer_run.json", "w") as fh:
    json.dump({
        "data": str(DATA_PATH),
        "bundles": {p.stem: {"path": str(p), "model_family": bundles[p.stem].get("model_family"),
                             "analysis_label": bundles[p.stem].get("analysis_label"),
                             "xgboost_version": bundles[p.stem].get("xgboost_version")}
                    for p in BUNDLE_PATHS},
        "endpoints": list(ENDPOINTS), "exclusion": EXCLUSION,
        "landmarks": list(LANDMARKS) if LANDMARKS else None,
        "horizons_days": list(HORIZONS), "n_bootstrap": N_BOOTSTRAP, "seed": SEED,
        "prep_stats": res["prep_stats"],
    }, fh, indent=2, default=str)
print(f"wrote outputs -> {OUTPUT_DIR}")

## Sanity checks: feature coverage and units

Check these before reading any metric. A covariate the local frame never produces is imputed with the
pooled mean for every patient, which carries no signal. A local mean more than 3× off the pooled mean
usually means a unit mismatch.

In [ ]:
cov = res["coverage"]
display(cov.head())
num = cov.select_dtypes("number").columns.difference(["landmark_days"])
cov.groupby(["bundle", "landmark_days", "config"])[list(num)].mean().round(3)

In [ ]:
uc = res["unit_check"]
flagged = uc[uc["unit_flag"]].sort_values("mean_ratio", ascending=False)
print(f"{flagged['covariate'].nunique()} covariate(s) flagged (>3x from pooled mean)")
flagged.drop_duplicates("covariate")

## Discrimination

In [ ]:
m = res["metrics"].copy()

def fmt_ci(est, lo, hi, d=3):
    if not np.isfinite(est):
        return ""
    return f"{est:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]" if np.isfinite(lo) else f"{est:.{d}f}"

for c in ("c_index_lo", "c_index_hi", "delta_c_vs_baseline", "delta_c_lo", "delta_c_hi"):
    if c not in m:
        m[c] = np.nan
m["C (95% CI)"] = [fmt_ci(r.c_index, r.c_index_lo, r.c_index_hi) for r in m.itertuples()]
m["ΔC vs age (95% CI)"] = [fmt_ci(r.delta_c_vs_baseline, r.delta_c_lo, r.delta_c_hi) for r in m.itertuples()]
m["HR/SD (95% CI)"] = [fmt_ci(r.hr_per_sd, r.hr_per_sd_lo, r.hr_per_sd_hi, 2) for r in m.itertuples()]

summary = m[m["config"] != "baseline"][[
    "endpoint", "landmark_days", "bundle", "n_patients", "n_events",
    "C (95% CI)", "ΔC vs age (95% CI)", "HR/SD (95% CI)", "uno_c", "mean_auc",
]].round({"uno_c": 3, "mean_auc": 3})
summary.sort_values(["endpoint", "landmark_days", "bundle"]).reset_index(drop=True)

In [ ]:
# Harrell's C (95% bootstrap CI) by landmark; one panel per endpoint, full vs age-only
plot = m.dropna(subset=["c_index"])
panels = [ep for ep in ENDPOINTS if ep in set(plot["endpoint"])]
stems = sorted(plot["bundle"].unique())
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # fixed categorical order
color = {s: COLORS[i % len(COLORS)] for i, s in enumerate(stems)}
lms = sorted(plot["landmark_days"].unique())
width = 0.8 / max(1, 2 * len(stems))

fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.6), sharey=True, squeeze=False)
for ax, ep in zip(axes[0], panels):
    d = plot[plot["endpoint"] == ep]
    k = 0
    for s in stems:
        for cfg, mk, fill in (("both", "o", True), ("baseline", "s", False)):
            r = d[(d["bundle"] == s) & (d["config"] == cfg)].set_index("landmark_days").reindex(lms)
            x = np.arange(len(lms)) + (k - (2 * len(stems) - 1) / 2) * width
            k += 1
            yerr = np.vstack([r["c_index"] - r["c_index_lo"], r["c_index_hi"] - r["c_index"]])
            ax.errorbar(x, r["c_index"], yerr=yerr, fmt=mk, ms=7, lw=1.5, capsize=0,
                        color=color[s], mfc=color[s] if fill else "white", mew=1.5,
                        label=f"{s} · {'full' if cfg == 'both' else 'age-only'}")
    ax.axhline(0.5, color="#8a8a86", lw=1, ls="--", zorder=0)
    ax.set_xticks(range(len(lms)), [f"{lm}d" for lm in lms])
    n_ev = d.groupby("landmark_days")["n_events"].first().reindex(lms)
    ax.set_title(f"{ep.upper()}  (events: {', '.join(str(int(v)) for v in n_ev.fillna(0))})", fontsize=10)
    ax.set_xlabel("landmark")
    ax.grid(axis="y", color="#e6e5df", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
axes[0][0].set_ylabel("Harrell's C")
axes[0][-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=8)
fig.suptitle("Platinum-trained federated models on local endpoints", fontsize=11)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "transfer_c_index.png", dpi=200, bbox_inches="tight")
plt.show()

## Time-dependent AUC

In [ ]:
auc = res["auc_t"]
if auc.empty:
    print("no estimable AUC(t) horizons")
else:
    display(auc.pivot_table(index=["endpoint", "landmark_days", "bundle", "config"],
                            columns="horizon_days", values="auc").round(3))

## Risk-score distributions

The feature frame does not depend on the endpoint, so the score distribution is the same for every
endpoint. The check that matters is whether local scores fall inside the range the pooled model was
fitted on.

In [ ]:
sc = res["scores"]
sc0 = sc[(sc["endpoint"] == ENDPOINTS[0]) & (sc["config"] == "both")]
sc0.groupby(["bundle", "landmark_days"])["risk_score"].describe().round(3)